Hybrid Search Langchain

In [16]:
%pip install --upgrade --quiet pinecone-client pinecone-text pinecone-notebook

Note: you may need to restart the kernel to use updated packages.


ERROR: Ignored the following versions that require a different python version: 0.3.2 Requires-Python >=3.8,<3.11; 0.3.3 Requires-Python >=3.8,<3.11; 0.3.4 Requires-Python >=3.8,<3.11; 0.4.0 Requires-Python >=3.8,<3.11; 0.4.1 Requires-Python >=3.8,<3.11
ERROR: Could not find a version that satisfies the requirement pinecone-notebook (from versions: none)

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for pinecone-notebook


In [5]:
api_key="pcsk_4ArNai_L5uPcyjNdi2LJpdhtLNfmGQnAvwnn5RTUs2gjDVycKRHNrdypDDifWaf4S1M72d"

In [6]:
from langchain_community.retrievers import  PineconeHybridSearchRetriever

In [9]:
%pip install --upgrade --quiet pinecone
import os 
from pinecone import Pinecone,ServerlessSpec
index_name="hybrid-search-langchain-pinecone"
pc=Pinecone(api_key=api_key)

#ctreate the index
if index_name not in pc.list_indexes():
    pc.create_index(
        name=index_name,
        dimension=384, #dimension of dense vector
        metric='dotproduct',##sparse value suported for dotproduct only
        spec=ServerlessSpec(cloud='aws',region="us-east-1"),
    )


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [10]:

index=pc.Index(index_name)
index



Index(host='https://hybrid-search-langchain-pinecone-eg4hae6.svc.aped-4627-b74a.pinecone.io')

In [13]:
## vector embedding and sparse matrix 
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Ashish Chaubey\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ashish Chaubey\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [18]:
%pip install --upgrade --quiet pinecone-text
from pinecone_text.sparse import BM25Encoder
bm25_encoder = BM25Encoder().default()
bm25_encoder


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
[nltk_data] Downloading package punkt_tab to C:\Users\Ashish
[nltk_data]     Chaubey\AppData\Roaming\nltk_data...


Note: you may need to restart the kernel to use updated packages.


[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to C:\Users\Ashish
[nltk_data]     Chaubey\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [20]:
sentences=[
    "Building intelligent, scalable, and AI-powered digital solutions with modern backend technologies",
    "Python Full Stack Developer focused on GenAI, FastAPI, scalable systems, and clean user experiences",
    "Passionate about transforming innovative ideas into high-performance real-world applications",
    
]
#tfidf value on these sentence
bm25_encoder.fit(sentences)
#store the value to a json file
bm25_encoder.dump("bm25_encoder.json")

  0%|          | 0/3 [00:00<?, ?it/s]

In [22]:
retriever = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25_encoder, index=index)

In [23]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x000001C70F01F560>, index=Index(host='https://hybrid-search-langchain-pinecone-eg4hae6.svc.aped-4627-b74a.pinecone.io'))

In [25]:
# prepare texts
texts = [
    "Building intelligent, scalable, and AI-powered digital solutions with modern backend technologies",
    "Python Full Stack Developer focused on GenAI, FastAPI, scalable systems, and clean user experiences",
    "Passionate about transforming innovative ideas into high-performance real-world applications",
]

# compute dense embeddings
embs = embeddings.embed_documents(texts)

# assemble payload and upsert using the keyword-only 'vectors' parameter
vectors = [
    {"id": f"doc-{i}", "values": embs[i], "metadata": {"text": texts[i]}}
    for i in range(len(texts))
]

index.upsert(vectors=vectors)

UpsertResponse(upserted_count=3)